In [0]:
from pathlib import Path
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

CONFIG_PATH = Path("/dbfs/mnt/mdm-database/config-script/config/data_loading_config.xlsx")

# MDM Mysqsl连接信息
MYSQL_HOST = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_HOST')
MYSQL_USER = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_USER')
MYSQL_PASSWORD = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_PSWD')
MYSQL_DRIVER = "com.mysql.cj.jdbc.Driver"
MYSQL_USE_SSL = True
DEFAULT_ID_KEY = "id"
DEFAULT_PARTITION_SIZE = 200000
DEFAULT_FETCH_SIZE = 10000
DEFAULT_STRING_PARTITIONS = 16
MAX_CONCURRENT_TABLES = 2
TARGET_RECORDS_PER_FILE = 500000
MIN_OUTPUT_PARTITIONS = 1
MAX_OUTPUT_PARTITIONS = 200

def build_jdbc_url(database: str) -> str:
    ssl_str = "&useSSL=true&enabledTLSProtocols=TLSv1.2" if MYSQL_USE_SSL else ""
    return (
        f"jdbc:mysql://{MYSQL_HOST}:3306/{database}"
        f"?serverTimezone=UTC&rewriteBatchedStatements=true&useUnicode=true&characterEncoding=UTF-8&zeroDateTimeBehavior=CONVERT_TO_NULL&useCompression=true{ssl_str}"
    )

def is_integer_pk(pk_type: str) -> bool:
    pk_type_lower = pk_type.strip().lower()
    return pk_type_lower.startswith(
        ("int", "bigint", "smallint", "tinyint", "mediumint")
    )

def read_mysql_int_pk(
    market: str,
    database: str,
    table_name: str,
    id_key: str = DEFAULT_ID_KEY,
    partition_size: int = DEFAULT_PARTITION_SIZE,
    fetch_size: int = DEFAULT_FETCH_SIZE,
    trace_id: str = "",
):
    # 整型主键：先取主键最小/最大值，再按范围并行拉取。
    print(f"{trace_id} loading {market} table: {database}.{table_name}, pk: {id_key}")

    mysql_url = build_jdbc_url(database)
    mysql_properties = {
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "driver": MYSQL_DRIVER,
    }

    edge_df = spark.read.jdbc(
        url=mysql_url,
        table=(
            f"(SELECT MIN({id_key}) as min_id, MAX({id_key}) as max_id "
            f"FROM {table_name}) as tmp"
        ),
        properties=mysql_properties,
    )
    edge_scope = edge_df.first()
    if edge_scope is None:
        return None, None, None

    lower_bound = edge_scope["min_id"]
    upper_bound = edge_scope["max_id"]
    if lower_bound is None or upper_bound is None:
        return None, None, None

    num_partitions = ((upper_bound - lower_bound) // partition_size) + 1
    print(
        f"{trace_id} lower_bound: {lower_bound}, upper_bound: {upper_bound}, "
        f"num_partitions: {num_partitions}"
    )

    df = (
        spark.read.format("jdbc")
        .option("url", mysql_url)
        .option("driver", MYSQL_DRIVER)
        .option("dbtable", table_name)
        .option("user", MYSQL_USER)
        .option("password", MYSQL_PASSWORD)
        .option("partitionColumn", id_key)
        .option("lowerBound", lower_bound)
        .option("upperBound", upper_bound)
        .option("numPartitions", num_partitions)
        .option("fetchSize", fetch_size)
        .load()
    )

    print(f"{trace_id} finished")
    return df, lower_bound, upper_bound


def read_mysql_string_pk(
    market: str,
    database: str,
    table_name: str,
    id_key: str = DEFAULT_ID_KEY,
    num_partitions: int = DEFAULT_STRING_PARTITIONS,
    fetch_size: int = DEFAULT_FETCH_SIZE,
    trace_id: str = "",
):
    # 字符串主键：用 CRC32 分桶生成 predicates，避免单线程全表扫描。
    print(f"{trace_id} loading {market} table: {database}.{table_name}, pk: {id_key}")

    mysql_url = build_jdbc_url(database)
    mysql_properties = {
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "driver": MYSQL_DRIVER,
        "fetchsize": str(fetch_size),
    }
    predicates = [
        f"MOD(CRC32({id_key}), {num_partitions}) = {idx}"
        for idx in range(num_partitions)
    ]

    df = spark.read.jdbc(
        url=mysql_url,
        table=table_name,
        predicates=predicates,
        properties=mysql_properties,
    )

    print(f"{trace_id} finished")
    return df, None, None


def clamp_partitions(value: int) -> int:
    return max(MIN_OUTPUT_PARTITIONS, min(MAX_OUTPUT_PARTITIONS, value))


def pick_output_partitions(row_estimate: int | None) -> int:
    # 根据估算行数控制输出分区，平衡写入并行度与小文件数量。
    if not row_estimate or row_estimate <= 0:
        return clamp_partitions(DEFAULT_STRING_PARTITIONS)
    target = (row_estimate // TARGET_RECORDS_PER_FILE) + 1
    return clamp_partitions(target)

def load_config() -> pd.DataFrame:
    if not CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Config not found at {CONFIG_PATH}. Update CONFIG_PATH if needed."
        )
    return pd.read_excel(CONFIG_PATH)


def process_table(item: dict) -> None:
    market = str(item.get("market", "")).strip()
    database = str(item.get("source_database", "")).strip()
    table = str(item.get("source_table", "")).strip()
    target_path = str(item.get("target_path", "")).strip()
    primary_key = str(item.get("primary_key", "")).strip()
    pk_type = str(item.get("pk_type", "")).strip()

    if not (market and database and table and target_path):
        return

    trace_id = f"[{market}-{database}-{table}]"

    try:
        if not primary_key:
            print(f"{trace_id} loading {market} table: {database}.{table}, pk: (none)")
            jdbc_url = build_jdbc_url(database)
            df = (
                spark.read.format("jdbc")
                .option("url", jdbc_url)
                .option("dbtable", table)
                .option("user", MYSQL_USER)
                .option("password", MYSQL_PASSWORD)
                .option("driver", MYSQL_DRIVER)
                .load()
            )
            lower_bound = None
            upper_bound = None
        elif is_integer_pk(pk_type):
            df, lower_bound, upper_bound = read_mysql_int_pk(
                market,
                database,
                table,
                primary_key,
                trace_id=trace_id,
            )
        else:
            df, lower_bound, upper_bound = read_mysql_string_pk(
                market,
                database,
                table,
                primary_key,
                trace_id=trace_id,
            )

        if df is None:
            print(f"{trace_id} [SKIP] market={market} database={database} table={table}")
            return

        # 整型主键场景可用主键范围粗略估算行数，用于写入前分区调整。
        row_estimate = None
        if lower_bound is not None and upper_bound is not None:
            row_estimate = max(0, int(upper_bound) - int(lower_bound) + 1)

        # 写 Delta 前先调分区，减少过多小文件或单任务过重。
        target_partitions = pick_output_partitions(row_estimate)
        current_partitions = df.rdd.getNumPartitions()
        if target_partitions < current_partitions:
            df = df.coalesce(target_partitions)
        elif target_partitions > current_partitions:
            df = df.repartition(target_partitions)

        (
            df.write.format("delta")
            .option("maxRecordsPerFile", TARGET_RECORDS_PER_FILE)
            .mode("overwrite")
            .save(target_path)
        )

        print(f"{trace_id} [OK] -> {target_path}")
    except Exception as exc:
        print(f"{trace_id} [ERROR] {type(exc).__name__}: {exc}")
        raise

def main() -> None:
    # spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
    # spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
    cfg = load_config()
    required_cols = {
        "market",
        "source_database",
        "source_table",
        "target_path",
        "is_active",
    }
    missing = required_cols - set(cfg.columns)
    if missing:
        raise ValueError(f"Config missing columns: {', '.join(sorted(missing))}")

    if "primary_key" not in cfg.columns:
        cfg["primary_key"] = ""
    if "pk_type" not in cfg.columns:
        cfg["pk_type"] = ""

    cfg = cfg[cfg["is_active"] == True]

    items = cfg[
        [
            "market",
            "source_database",
            "source_table",
            "target_path",
            "primary_key",
            "pk_type",
        ]
    ].to_dict("records")

    # 表级并发：单表失败不阻断其它任务，末尾统一汇总失败数。
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_TABLES) as executor:
        futures = {
            executor.submit(process_table, item): item
            for item in items
        }
        failed = 0
        for future in as_completed(futures):
            item = futures[future]
            trace_id = (
                f"[{item.get('market', '')}-{item.get('source_database', '')}-"
                f"{item.get('source_table', '')}]"
            )
            try:
                future.result()
            except Exception as exc:
                failed += 1
                print(f"{trace_id} [FAILED] {type(exc).__name__}: {exc}")

        if failed > 0:
            raise RuntimeError(f"Load completed with {failed} failed table tasks")

# start loading
main()
